# XGBoost — Walmart Store Sales Forecasting

In [1]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.environ["WALMART_ROOT"] = "/content/drive/MyDrive/MLFinalAssignment"

Mounted at /content/drive


In [ ]:
import importlib.util
import os
import pathlib
import subprocess
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    missing = [p for p in ("xgboost", "mlflow", "dagshub")
               if importlib.util.find_spec(p) is None]
    if missing:
        print("installing", *missing)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    if not pathlib.Path("/content/drive").exists():
        from google.colab import drive
        drive.mount("/content/drive")


def find_root() -> pathlib.Path:
    """Locate the repo: it must hold train.csv and src/walmart_prep.py."""
    candidates = []
    if os.environ.get("WALMART_ROOT"):
        candidates.append(pathlib.Path(os.environ["WALMART_ROOT"]))
    candidates += [pathlib.Path.cwd(), pathlib.Path.cwd().parent]
    if IN_COLAB:
        drive_root = pathlib.Path("/content/drive/MyDrive")
        candidates += [drive_root / "MLFinalProject", pathlib.Path("/content/MLFinalProject")]
        if drive_root.exists():
            candidates += sorted(p for p in drive_root.glob("*") if (p / "train.csv").exists())
    for c in candidates:
        if (c / "train.csv").exists() and (c / "src" / "walmart_prep.py").exists():
            return c.resolve()
    raise FileNotFoundError(
        "Could not find the project. Set WALMART_ROOT to the folder containing "
        "train.csv and src/walmart_prep.py, then re-run this cell.\n"
        f"Looked in: {[str(c) for c in candidates]}")


ROOT = find_root()
sys.path.insert(0, str(ROOT / "src"))
for sub in ("docs", "submissions"):
    (ROOT / sub).mkdir(exist_ok=True)

print(f"IN_COLAB={IN_COLAB}\nROOT={ROOT}")

installing mlflow dagshub
IN_COLAB=True
ROOT=/content/drive/MyDrive/MLFinalAssignment


In [ ]:
import dagshub
import mlflow

dagshub.init(repo_owner='smama23', repo_name='MLFinalProject', mlflow=True)

EXPERIMENT_NAME = 'XGBoost_Training'
REGISTERED_MODEL_NAME = 'WalmartSalesForecast'

mlflow.set_experiment(EXPERIMENT_NAME)

if mlflow.active_run() is not None:
    mlflow.end_run()

print('Tracking URI:', mlflow.get_tracking_uri())
print('Experiment  :', EXPERIMENT_NAME)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=b52167f1-69a3-46ad-9eb6-efc03d85b899&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=a65733aeae8ee3b79a378ed8dc1fade7dc4df805b53fc9f36961e40add15b519




Accessing as smama23

Initialized MLflow to track repo "smama23/MLFinalProject"

Repository smama23/MLFinalProject initialized!

2026/07/10 17:03:20 INFO mlflow.tracking.fluent: Experiment with name 'XGBoost_Training' does not exist. Creating a new experiment.


Tracking URI: https://dagshub.com/smama23/MLFinalProject.mlflow
Experiment  : XGBoost_Training


In [ ]:
import time

import numpy as np
import pandas as pd

import xgboost as xgb
import mlflow
import mlflow.sklearn
from sklearn.pipeline import Pipeline

from walmart_prep import (
    FOLDS, HORIZON, MARKDOWN_COLS, MIN_SAFE_LAG,
    SeasonalResidualRegressor, WalmartFeatureBuilder,
    december_shape_report, load_raw, make_submission,
    score_fold, seasonal_naive, wmae_weights,
)

EXPERIMENT = EXPERIMENT_NAME
print("tracking:", mlflow.get_tracking_uri())

train, test, features, stores = load_raw(str(ROOT))
SEED = 0

print(f"train {train.shape}   test {test.shape}")
print(f"train {train.Date.min().date()} .. {train.Date.max().date()}")
print(f"test  {test.Date.min().date()} .. {test.Date.max().date()}")
print(f"horizon = {HORIZON} weeks  ->  minimum safe lag = {MIN_SAFE_LAG} weeks")

tracking: https://dagshub.com/smama23/MLFinalProject.mlflow
train (421570, 5)   test (115064, 4)
train 2010-02-05 .. 2012-10-26
test  2012-11-02 .. 2013-07-26
horizon = 39 weeks  ->  minimum safe lag = 39 weeks


In [ ]:
USE_GPU = True 


def device_params() -> dict:
    """Probe CUDA with the objective we actually use; fall back to CPU on any failure."""
    if USE_GPU:
        try:
            xgb.XGBRegressor(device="cuda", tree_method="hist", n_estimators=2,
                             objective="reg:absoluteerror").fit(np.zeros((16, 2)), np.arange(16.0))
            print("using device=cuda")
            return {"device": "cuda", "tree_method": "hist"}
        except Exception as e:
            print("GPU unavailable, CPU fallback:", str(e).splitlines()[0][:80])
    return {"device": "cpu", "tree_method": "hist"}


DEVICE_PARAMS = device_params()
DEVICE = DEVICE_PARAMS["device"]
print(f"device={DEVICE}  cores={os.cpu_count()}")
DEVICE_PARAMS

using device=cuda
device=cuda  cores=2


{'device': 'cuda', 'tree_method': 'hist'}

## `XGBoost_Cleaning`

In [ ]:
with mlflow.start_run(run_name="XGBoost_Cleaning"):
    mlflow.log_param("device", DEVICE)
    mlflow.log_params({
        "internal_gaps": "fill 0 (94.5% of sales next to a gap are < $500)",
        "markdowns": "fill 0 + markdown_era flag (two different missingness mechanisms)",
        "cpi_unemployment": "ffill per store, then deviation from the store's train mean",
        "negative_sales": "kept (0.305% of rows, genuine returns/adjustments)",
        "target_transform": "none (WMAE is L1; log1p would reshape the loss)",
        "store_dept_encoding": "numeric",
    })

    counts = train.groupby(["Store", "Dept"]).size()
    span = train.groupby(["Store", "Dept"]).Date.agg(["min", "max"])
    span["weeks"] = (span["max"] - span["min"]).dt.days // 7 + 1
    gap_cells = int((span["weeks"] - counts).clip(lower=0).sum())
    tr_pairs = set(map(tuple, train[["Store", "Dept"]].drop_duplicates().values))
    te_pairs = set(map(tuple, test[["Store", "Dept"]].drop_duplicates().values))
    cold = sorted(te_pairs - tr_pairs)

    stats = {
        "train_rows": len(train), "test_rows": len(test),
        "train_pairs": len(tr_pairs), "test_pairs": len(te_pairs),
        "cold_start_pairs": len(cold),
        "negative_sales_frac": float((train.Weekly_Sales < 0).mean()),
        "internal_gap_cells": gap_cells,
        "markdown_era_share_train": float((train.Date >= pd.Timestamp("2011-11-11")).mean()),
        "holiday_weight_share_test": float(
            5 * test.IsHoliday.sum() / wmae_weights(test.IsHoliday).sum()),
    }
    mlflow.log_metrics(stats)

pd.Series(stats).to_frame("value")

🏃 View run XGBoost_Cleaning at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/1/runs/10e0f7761d434e1bab91d693c706b17b
🧪 View experiment at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/1


,value
train_rows,421570.000000
test_rows,115064.000000
train_pairs,3331.000000
test_pairs,3169.000000
cold_start_pairs,11.000000
negative_sales_frac,0.003048
internal_gap_cells,27667.000000
markdown_era_share_train,0.359210
holiday_weight_share_test,0.296068


## `XGBoost_Baseline`

In [ ]:
baseline_scores = {}
with mlflow.start_run(run_name="XGBoost_Baseline"):
    mlflow.log_param("model", "seasonal naive: lag-52, fallback pair median, then global median")
    for fold in FOLDS:
        tr = train[train.Date <= fold.cut]
        va = train[(train.Date >= fold.val_start) & (train.Date <= fold.val_end)]
        s = score_fold(va.Weekly_Sales, seasonal_naive(tr, va), va)
        baseline_scores[fold.name] = s
        mlflow.log_metric(f"wmae_{fold.name}", s["wmae"])
        for k, v in s.items():
            if k.startswith("mae_"):
                mlflow.log_metric(f"{k}_{fold.name}", v)

BASELINE = {k: v["wmae"] for k, v in baseline_scores.items()}
pd.DataFrame(baseline_scores).round(1)

🏃 View run XGBoost_Baseline at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/1/runs/605ec5f648fb44cbaba7ddc0c37a56e7
🧪 View experiment at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/1


,mirror,recent,early
wmae,2037.8,1807.2,2018.6
mae,1949.9,1805.5,1955.9
mae_holiday,2320.2,1815.7,2170.0
mae_nonholiday,1918.6,1804.9,1931.2
bias,-381.1,-375.3,-341.3
mae_thanksgiving,2387.6,NaN,2399.5
mae_christmas,2711.7,NaN,2721.0
mae_superbowl,1860.7,1832.5,1874.9
mae_laborday,NaN,1798.6,1675.8


## `XGBoost_CV`

In [ ]:
DEFAULT_PARAMS = dict(
    objective="reg:absoluteerror",
    n_estimators=900,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=5,
    subsample=1.0,
    colsample_bytree=1.0,
    reg_lambda=1.0,
    max_bin=256,
    n_jobs=-1,
    random_state=SEED,
    verbosity=0,
    **DEVICE_PARAMS,
)

FOLD_DATA = {}
for fold in FOLDS:
    t0 = time.time()
    tr = train[train.Date <= fold.cut]
    va = train[(train.Date >= fold.val_start) & (train.Date <= fold.val_end)]
    fb = WalmartFeatureBuilder(features, stores)
    Xtr = fb.fit_transform(tr.drop(columns=["Weekly_Sales"]), tr.Weekly_Sales)
    Xva = fb.transform(va.drop(columns=["Weekly_Sales"]))
    FOLD_DATA[fold.name] = dict(fb=fb, Xtr=Xtr, Xva=Xva, va=va,
                                ytr=tr.Weekly_Sales.to_numpy(),
                                w=wmae_weights(tr.IsHoliday))
    print(f"{fold.name:7s} {tr.Date.nunique():3d} train weeks | {Xtr.shape[1]:2d} features "
          f"| {time.time() - t0:4.1f}s | xmas seasons={fb.n_xmas_seasons_} "
          f"| dropped: {fb.dropped_columns_ or 'none'}")


def fit_fold(fold_name, params, drop=()):
    """Fit the residual learner on a cached fold. Returns (model, scores)."""
    d = FOLD_DATA[fold_name]
    keep = [c for c in d["Xtr"].columns if c not in drop]
    model = SeasonalResidualRegressor(xgb.XGBRegressor(**params))
    model.fit(d["Xtr"][keep], d["ytr"], sample_weight=d["w"])
    pred = model.predict(d["Xva"][keep])
    return model, score_fold(d["va"].Weekly_Sales, pred, d["va"])

mirror   91 train weeks | 42 features |  5.0s | xmas seasons=1 | dropped: ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'lag_104', 'markdown_era', 'md_any', 'md_total', 'yoy_trend']
recent  104 train weeks | 51 features |  6.2s | xmas seasons=2 | dropped: ['lag_104']
early    78 train weeks | 42 features |  3.7s | xmas seasons=1 | dropped: ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'lag_104', 'markdown_era', 'md_any', 'md_total', 'yoy_trend']


In [ ]:
cv_scores = {}
with mlflow.start_run(run_name="XGBoost_CV"):
    mlflow.log_param("device", DEVICE)
    mlflow.log_params({**{k: v for k, v in DEFAULT_PARAMS.items() if k != "callbacks"},
                       "target": "residual (y - seasonal baseline)",
                       "sample_weight": "5 on holiday weeks"})
    for fold in FOLDS:
        _, s = fit_fold(fold.name, DEFAULT_PARAMS)
        cv_scores[fold.name] = s
        mlflow.log_metric(f"wmae_{fold.name}", s["wmae"])
        for k, v in s.items():
            if k.startswith("mae_"):
                mlflow.log_metric(f"{k}_{fold.name}", v)
        gain = 100 * (1 - s["wmae"] / BASELINE[fold.name])
        mlflow.log_metric(f"gain_vs_naive_{fold.name}", gain)
        print(f"{fold.name:7s} naive {BASELINE[fold.name]:7.1f} -> XGB {s['wmae']:7.1f}  ({gain:+.1f}%)")
    mlflow.log_metric("wmae_mean", float(np.mean([s["wmae"] for s in cv_scores.values()])))

REF = {k: v["wmae"] for k, v in cv_scores.items()}
pd.DataFrame(cv_scores).round(1)

mirror  naive  2037.8 -> XGB  1843.2  (+9.6%)
recent  naive  1807.2 -> XGB  1630.9  (+9.8%)
early   naive  2018.6 -> XGB  1870.1  (+7.4%)
🏃 View run XGBoost_CV at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/1/runs/57d6ce98910941b9b77e9f71e0404d73
🧪 View experiment at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/1


,mirror,recent,early
wmae,1843.2,1630.9,1870.1
mae,1787.7,1637.1,1809.9
mae_holiday,2021.2,1601.1,2015.6
mae_nonholiday,1768.0,1639.0,1786.2
bias,49.6,-197.7,-301.3
mae_thanksgiving,2180.9,NaN,2580.3
mae_christmas,2306.9,NaN,2099.1
mae_superbowl,1574.6,1565.6,1682.5
mae_laborday,NaN,1637.1,1692.1


## `XGBoost_Feature_Selection`

In [ ]:
BLOCKS = {
    "drop_time_index": ["t", "year"],
    "drop_markdowns": MARKDOWN_COLS + ["md_total", "md_any", "markdown_era"],
    "drop_exogenous": ["Temperature", "Fuel_Price", "CPI_dev", "Unemployment_dev"],
    "drop_xmas_aligned_lag": ["xmas_aligned_lag"],
    "drop_shared_strength": ["store_lag_52", "dept_lag_52", "pair_share_of_store",
                             "Size_per_dept_sale"],
}

rows = [{"ablation": "keep everything", **{f.name: REF[f.name] for f in FOLDS}}]
with mlflow.start_run(run_name="XGBoost_Feature_Selection"):
    mlflow.log_param("device", DEVICE)
    mlflow.log_metrics({f"wmae_{k}_reference": v for k, v in REF.items()})

    m, _ = fit_fold("mirror", DEFAULT_PARAMS)
    booster = m.estimator_.get_booster()
    gain = booster.get_score(importance_type="gain")
    imp = (pd.Series(gain).reindex(FOLD_DATA["mirror"]["Xtr"].columns)
             .fillna(0.0).sort_values(ascending=False))
    path = ROOT / "docs" / "xgboost_gain_importance.csv"
    imp.to_frame("gain").to_csv(path)
    mlflow.log_artifact(str(path))
    print("top 12 features by gain:\n" + imp.head(12).round(1).to_string() + "\n")

    for tag, cols in BLOCKS.items():
        with mlflow.start_run(run_name=f"XGBoost_Feature_Selection__{tag}", nested=True):
            mlflow.log_param("dropped_columns", ", ".join(cols))
            row = {"ablation": tag}
            for fold in FOLDS:
                present = [c for c in cols if c in FOLD_DATA[fold.name]["Xtr"].columns]
                if not present:
                    row[fold.name] = np.nan
                    continue
                _, s = fit_fold(fold.name, DEFAULT_PARAMS, drop=present)
                row[fold.name] = s["wmae"]
                mlflow.log_metric(f"wmae_{fold.name}", s["wmae"])
                mlflow.log_metric(f"delta_{fold.name}", s["wmae"] - REF[fold.name])
            rows.append(row)
            print(f"{tag:24s} " + "  ".join(
                f"{f.name}={row[f.name]:7.1f}" if np.isfinite(row.get(f.name, np.nan))
                else f"{f.name}=    n/a" for f in FOLDS))

ablation = pd.DataFrame(rows).set_index("ablation")
delta = ablation.subtract(pd.Series(REF), axis=1).drop(index="keep everything")
print("\nWMAE change vs keeping everything (positive = the block was helping):")
delta.round(1)

top 12 features by gain:
t                       270.6
pre_xmas_days            96.0
Dept                     88.9
lag_45                   62.6
Size                     59.1
roll_mean_lag39_51       58.6
lag_39                   57.4
Type                     56.1
woy                      48.7
days_to_thanksgiving     45.2
Store                    40.1
is_easter_week           40.0

drop_time_index          mirror= 2025.4  recent= 1651.2  early= 2365.0
🏃 View run XGBoost_Feature_Selection__drop_time_index at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/1/runs/7eb9e6c9faf843339ccfad7e2c08961f
🧪 View experiment at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/1
drop_markdowns           mirror=    n/a  recent= 1647.5  early=    n/a
🏃 View run XGBoost_Feature_Selection__drop_markdowns at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/1/runs/c1a5b004e28a4789be09659709905295
🧪 View experiment at: https://dagshub.com/smama23/MLFinalProj

,mirror,recent,early
ablation,,,
drop_time_index,182.2,20.3,494.9
drop_markdowns,NaN,16.5,NaN
drop_exogenous,7.0,9.2,110.5
drop_xmas_aligned_lag,-22.6,1.5,9.1
drop_shared_strength,10.5,40.5,65.0


## `XGBoost_Tuning`

In [ ]:
rng = np.random.default_rng(SEED)
GRID = {
    "learning_rate": [0.03, 0.05, 0.08],
    "max_depth": [6, 8, 10, 12],
    "min_child_weight": [1, 5, 20, 50],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "subsample": [0.7, 0.9, 1.0],
    "reg_lambda": [0.0, 1.0, 5.0],
    "n_estimators": [600, 900, 1200, 1500],
    "max_bin": [64, 128, 256],
}
N_TRIALS = 12


def sample_params():
    p = dict(DEFAULT_PARAMS)
    for k, v in GRID.items():
        p[k] = type(v[0])(rng.choice(v))
    return p


candidates = [("default", dict(DEFAULT_PARAMS))]
candidates += [(f"trial_{i:02d}", sample_params()) for i in range(N_TRIALS)]

trials = []
with mlflow.start_run(run_name="XGBoost_Tuning"):
    mlflow.log_params({"n_trials": N_TRIALS, "search": "random + incumbent",
                       "selection_fold": "mirror", "device": DEVICE})

    for name, params in candidates:
        t0 = time.time()
        if name == "default":
            s = cv_scores["mirror"]
        else:
            _, s = fit_fold("mirror", params)
        with mlflow.start_run(run_name=f"XGBoost_Tuning__{name}", nested=True):
            mlflow.log_params(params)
            mlflow.log_metric("wmae_mirror", s["wmae"])
            if "mae_christmas" in s:
                mlflow.log_metric("mae_christmas_mirror", s["mae_christmas"])
            mlflow.log_metric("fit_seconds", time.time() - t0)
        trials.append({"name": name, "params": params, "mirror": s["wmae"]})
        print(f"{name:12s} wmae_mirror={s['wmae']:7.1f}  ({time.time() - t0:4.0f}s)  "
              f"lr={params['learning_rate']} depth={params['max_depth']} "
              f"n={params['n_estimators']} sub={params['subsample']} col={params['colsample_bytree']}")

    top = sorted(trials, key=lambda r: r["mirror"])[:3]
    print("\nconfirming the top 3 on every fold...")
    for r in top:
        for fold in FOLDS:
            if fold.name == "mirror":
                continue
            elif r["name"] == "default":
                r[fold.name] = REF[fold.name]
            else:
                r[fold.name] = fit_fold(fold.name, r["params"])[1]["wmae"]
        r["mean"] = float(np.mean([r[f.name] for f in FOLDS]))
        r["beats_naive"] = all(r[f.name] < BASELINE[f.name] for f in FOLDS)
        print(f"  {r['name']:12s} " + "  ".join(f"{f.name}={r[f.name]:7.1f}" for f in FOLDS)
              + f"  mean={r['mean']:7.1f}  beats_naive={r['beats_naive']}")

    eligible = [r for r in top if r["beats_naive"]] or top
    best = min(eligible, key=lambda r: r["mean"])
    BEST_PARAMS = best["params"]
    mlflow.log_params({f"best_{k}": v for k, v in BEST_PARAMS.items()})
    mlflow.log_metrics({f"best_wmae_{f.name}": best[f.name] for f in FOLDS})
    mlflow.log_metric("best_wmae_mean", best["mean"])
    mlflow.log_param("best_candidate", best["name"])

print(f"\nwinner: {best['name']}   mean WMAE {best['mean']:.1f}")
print(f"default mean WMAE {np.mean(list(REF.values())):.1f}")
BEST_PARAMS

🏃 View run XGBoost_Tuning__default at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/1/runs/35b625a17aad4eb0887f08a6ed569cde
🧪 View experiment at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/1
default      wmae_mirror= 1843.2  (   1s)  lr=0.05 depth=8 n=900 sub=1.0 col=1.0
🏃 View run XGBoost_Tuning__trial_00 at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/1/runs/87084a4a58714de5b98dff0f77ba303a
🧪 View experiment at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/1
trial_00     wmae_mirror= 1889.0  (   9s)  lr=0.08 depth=10 n=600 sub=0.7 col=0.6
🏃 View run XGBoost_Tuning__trial_01 at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/1/runs/adcc5500e38341b8886ea91b6a1a72f0
🧪 View experiment at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/1
trial_01     wmae_mirror= 1839.9  (  35s)  lr=0.03 depth=12 n=1500 sub=0.9 col=1.0
🏃 View run XGBoost_Tuning__trial_02 at: https://dagshub.com

{'objective': 'reg:absoluteerror',
 'n_estimators': 1500,
 'learning_rate': 0.03,
 'max_depth': 10,
 'min_child_weight': 50,
 'subsample': 0.9,
 'colsample_bytree': 0.8,
 'reg_lambda': 5.0,
 'max_bin': 256,
 'n_jobs': -1,
 'random_state': 0,
 'verbosity': 0,
 'device': 'cuda',
 'tree_method': 'hist'}

## `XGBoost_Final`

In [ ]:
with mlflow.start_run(run_name="XGBoost_Final") as final_run:
    mlflow.log_param("device", DEVICE)
    mlflow.log_params({**BEST_PARAMS, "trained_on": "all 143 weeks",
                       "target": "residual (y - seasonal baseline)",
                       "selected_candidate": best["name"]})
    mlflow.log_metrics({f"cv_wmae_{f.name}": best[f.name] for f in FOLDS})
    mlflow.log_metric("cv_wmae_mean", best["mean"])

    pipe = Pipeline([
        ("features", WalmartFeatureBuilder(features, stores)),
        ("model", SeasonalResidualRegressor(xgb.XGBRegressor(**BEST_PARAMS))),
    ])
    pipe.fit(train.drop(columns=["Weekly_Sales"]), train.Weekly_Sales,
             model__sample_weight=wmae_weights(train.IsHoliday))

    y_pred = pipe.predict(test)
    print(f"predictions: n={len(y_pred)}  mean={y_pred.mean():.1f}  "
          f"min={y_pred.min():.1f}  max={y_pred.max():.1f}")

    fb = pipe.named_steps["features"]
    frame, verdict = december_shape_report(train, test, y_pred, fb.xmas_profile_)
    print(f"\nprofile fitted on {fb.n_xmas_seasons_} December season(s)")
    print(frame.to_string(index=False, float_format=lambda v: f"{v:8.3f}"))
    print(f"\npredicted peak {verdict['peak_predicted']} | implied peak {verdict['peak_implied']}")
    print("GATE:", "PASS" if verdict["passed"] else "REVIEW -- " + "; ".join(verdict["problems"]))

    gate_path = ROOT / "docs" / "xgboost_december_gate.csv"
    frame.to_csv(gate_path, index=False)
    mlflow.log_artifact(str(gate_path))
    mlflow.log_metric("december_xmas_gap_pct", verdict["xmas_gap_pct"])
    mlflow.log_metric("december_gate_passed", int(verdict["passed"]))
    mlflow.log_param("december_peak_predicted", verdict["peak_predicted"])

    logged_model = mlflow.sklearn.log_model(
        pipe, name="pipeline",
        code_paths=[str(ROOT / "src" / "walmart_prep.py")],
        input_example=test.head(3),
        serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
    )

    sub = make_submission(test, y_pred, ROOT / "submissions" / "xgboost.csv")
    mlflow.log_artifact(str(ROOT / "submissions" / "xgboost.csv"))
    FINAL_RUN_ID = final_run.info.run_id

MODEL_URI = logged_model.model_uri
print(f"\nrun_id    = {FINAL_RUN_ID}")
print(f"model_uri = {MODEL_URI}")
sub.head()

predictions: n=115064  mean=16651.5  min=-3881.6  max=628804.9

profile fitted on 2 December season(s)
      week  k_end  predicted  implied  gap_pct
2012-11-30    -25      0.991    0.969    2.228
2012-12-07    -18      1.110    1.109    0.096
2012-12-14    -11      1.176    1.208   -2.646
2012-12-21     -4      1.443    1.489   -3.091
2012-12-28      3      1.144    1.243   -7.925
2013-01-04     10      0.960    0.919    4.419

predicted peak 2012-12-21 | implied peak 2012-12-21
GATE: PASS


2026/07/10 17:25:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling 

🏃 View run XGBoost_Final at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/1/runs/29b61d3766fb4eb4bb29173295e7d69b
🧪 View experiment at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/1

run_id    = 29b61d3766fb4eb4bb29173295e7d69b
model_uri = models:/m-fcb8dcef2f0c4c9a90b9d21a6de4120c


,Id,Weekly_Sales
0,1_1_2012-11-02,38229.738159
1,1_1_2012-11-09,21174.481201
2,1_1_2012-11-16,20299.196655
3,1_1_2012-11-23,21254.922119
4,1_1_2012-11-30,25428.642700


In [ ]:
loaded = mlflow.sklearn.load_model(MODEL_URI)
reloaded_pred = loaded.predict(test)
print("max |reloaded - original| =", float(np.abs(reloaded_pred - y_pred).max()))
assert np.allclose(reloaded_pred, y_pred, atol=1e-3), "the logged Pipeline must reproduce its predictions"
print("Pipeline round-trip OK")

mv = mlflow.register_model(MODEL_URI, REGISTERED_MODEL_NAME)
print(f"registered {mv.name} version {mv.version}")
try:
    from mlflow import MlflowClient
    MlflowClient().set_registered_model_alias(mv.name, "xgboost", mv.version)
    print(f"alias 'xgboost' -> version {mv.version}")
except Exception as e:
    print("alias not set (optional):", str(e)[:80])

max |reloaded - original| = 0.0
Pipeline round-trip OK


Registered model 'WalmartSalesForecast' already exists. Creating a new version of this model...
2026/07/10 17:26:08 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: WalmartSalesForecast, version 3
Created version '3' of model 'WalmartSalesForecast'.


registered WalmartSalesForecast version 3
alias 'xgboost' -> version 3


In [ ]:
print("experiments: https://dagshub.com/smama23/MLFinalProject/experiments")
print("registry   : https://dagshub.com/smama23/MLFinalProject/models")
try:
    runs = mlflow.search_runs(experiment_names=[EXPERIMENT])
    cols = [c for c in ("tags.mlflow.runName", "metrics.wmae_mirror", "metrics.wmae_recent",
                        "metrics.wmae_early", "metrics.december_gate_passed") if c in runs.columns]
    summary = runs[cols]
    try:
        display(summary)
    except NameError:
        print(summary.to_string(index=False))
except Exception as e:
    print("could not fetch run summary (model is already logged):", e)

experiments: https://dagshub.com/smama23/MLFinalProject/experiments
registry   : https://dagshub.com/smama23/MLFinalProject/models


,tags.mlflow.runName,metrics.wmae_mirror,metrics.wmae_recent,metrics.wmae_early,metrics.december_gate_passed
0,XGBoost_Final,NaN,NaN,NaN,1.0
1,XGBoost_Final,NaN,NaN,NaN,1.0
2,XGBoost_Tuning__trial_11,1878.051113,NaN,NaN,NaN
3,XGBoost_Tuning__trial_10,1912.856708,NaN,NaN,NaN
4,XGBoost_Tuning__trial_09,1832.197125,NaN,NaN,NaN
5,XGBoost_Tuning__trial_08,1874.946040,NaN,NaN,NaN
6,XGBoost_Tuning__trial_07,1866.670720,NaN,NaN,NaN
7,XGBoost_Tuning__trial_06,1817.694147,NaN,NaN,NaN
8,XGBoost_Tuning__trial_05,1837.403563,NaN,NaN,NaN
9,XGBoost_Tuning__trial_04,1877.110011,NaN,NaN,NaN
